# Eksplorasi: Load dan Chunk Dokumen Besar

Notebook ini mendemonstrasikan cara memuat dokumen PDF besar menggunakan metode **Lazy Loading** agar efisien dalam penggunaan memori, lalu memotongnya (*chunking*) menggunakan **RecursiveCharacterTextSplitter** dengan menjaga agar metadata (seperti nomor halaman) tidak hilang.

In [10]:
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Ganti dengan nama file PDF yang Anda letakkan di data/raw/
# Jika menggunakan Absolute Path di Windows, WAJIB tambahkan huruf 'r' di depan tanda kutip
# agar backslash (\) tidak terbaca sebagai escape character (seperti \n atau \a)
pdf_path = r"D:\Self Project\SumoPod API\Profile\projects\ai_document_assistant\data\raw\ADPU441003 - Kebijakan Publik.pdf"

### 1. Lazy Loading
Kita memuat dokumen menggunakan `lazy_load()`. Metode ini mengembalikan *iterator* alih-alih meletakkan semua halaman ke memori secara bersamaan.

In [11]:
try:
    loader = PyMuPDFLoader(pdf_path)
    
    # Kita asumsikan kita hanya ingin melihat dan memproses 3 halaman pertama sebagai contoh
    lazy_docs = loader.lazy_load()
    
    sample_pages = []
    for i, doc in enumerate(lazy_docs):
        sample_pages.append(doc)
        if i >= 2: # Berhenti setelah mengambil 3 halaman
            break
            
    print(f"Berhasil memuat {len(sample_pages)} halaman awal secara lazy.")
except FileNotFoundError:
    print(f"File {pdf_path} tidak ditemukan. Silakan taruh file PDF di folder data/raw/ terlebih dahulu.")
except Exception as e:
    print(f"Gagal memuat dokumen: {e}")

Berhasil memuat 3 halaman awal secara lazy.


### 2. Semantic Chunking
Membagi teks ke dalam chunk kecil yang aman untuk konteks LLM.

In [12]:
if 'sample_pages' in locals() and len(sample_pages) > 0:
    # Kita atur ukuran chunk 1000 karakter, dengan overlap 200 karakter antar chunk
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=1000,
        chunk_overlap=200,
        length_function=len,
        is_separator_regex=False,
    )
    
    # Memecah halaman-halaman yang kita ambil tadi
    chunks = text_splitter.split_documents(sample_pages)
    print(f"Dari {len(sample_pages)} halaman, berhasil dipecah menjadi {len(chunks)} chunks.\n")
    
    # Menampilkan chunk pertama beserta metadatanya
    print("--- Chunk Pertama ---")
    print(chunks[0].page_content)
    print("\n--- Metadata Chunk Pertama ---")
    print(chunks[0].metadata)
else:
    print("Teks belum siap di-chunk. Pastikan file PDF berhasil diload.")

Dari 3 halaman, berhasil dipecah menjadi 3 chunks.

--- Chunk Pertama ---
Daftar Isi a 
// 
Tinjauan Mata Kuliah | 
vii 
Modul 01 
[a5 
Definisi dan Makna Kebij akan Publik 
Kegiatan Belajar 
1 
Definisi dan Makna Kebij akan Publik 
Kegiatan Belajar 2 | 1.16 
Kebij akan Publik dan Kepentingan 
Publik 
Kegiatan Belajar 3 
Tipologi Kebij akan Publik 
Modul 02 
Model/ Pendekatan Kebij akan Publik 
Kegiatan Belajar 
1 
Beberapa Model/ Pendekatan 
Kebijakan Publik 
Kegiatan Belajar 2 | 2.19 
Pendekat 
an dalam Analisis Kebij akan 
Publik 
Kegiatan Belajar 3 
Kajian Kebij akan Publik Deliberatif 
Modul 03 
Proses Kebij akan Publik 
Kegiatan Belajar 
1 
Makna Kebij akan Publik sebagai Proses 
Kegiatan Belajar 2 
Proses Teknokratis dan Demokratis 
20008565_ADPU4410_EDISI3_1SLindb 3 
1/3/2023 3:06:16 PM

--- Metadata Chunk Pertama ---
{'producer': 'pikepdf 8.7.1', 'creator': 'ocrmypdf 15.2.0+dfsg1 / Tesseract OCR-PDF 5.3.4', 'creationdate': '2026-02-10T23:45:54+00:00', 'source': 'D:\\Self Proje